# fea-surrogate — SFEM training

Trains a per-node stress surrogate on [`cmudrc/SFEM`](https://huggingface.co/datasets/cmudrc/SFEM)
and exports weights plus a conformal calibration record, which is what
`SurrogateScreening/v1` refuses documents without.

**Runtime → Change runtime type → T4 GPU** before running. Cells run 1 → 10 in
order; each depends only on the ones above it.

### Two decisions baked in

**The target is dimensionless.** SFEM solves metre-scale parts in a soft polymer
(E = 2.303 GPa, ν = 0.4002) under 200–20000 N point loads, giving stresses of a
few Pa. Real designs are millimetre-scale steel at hundreds of MPa — six orders
away, so a model emitting absolute Pa would have a domain envelope nothing you
design ever enters.

In linear elasticity the stress field is **independent of Young's modulus** — E
only sets displacement — and stress scales as F/L². So training on

    tau = sigma * L^2 / F

learns the shape function itself, which transfers to any scale, any load and any
stiffness. What does *not* transfer is Poisson's ratio, because the fixed facets
are displacement boundary conditions; ν = 0.4002 is therefore a declared bound in
the exported domain rather than something to wave away.

**Calibration is multiplicative.** Peak stress spans five orders of magnitude, so
a constant-width band is far too wide at the bottom and too tight at the top even
when its average coverage is exactly right. Conformal is applied to log residuals
instead, giving a `×/÷ factor` interval, and coverage is reported per quintile so
conditional failure cannot hide behind a good marginal number.


## 1 · Setup

In [ ]:
import os, sys, subprocess, pathlib, json, hashlib, datetime
try:
    import torch_geometric  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch-geometric", "huggingface_hub"], check=True)
import torch, numpy as np, torch.nn as nn
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4")

## 2 · Data

Only `Pytorch_Geometric_Files/processed` — 11.9 GB. The raw tars and STEP files
are not needed: the processed files are the output of the authors' `process()`
over them.

They carry **von Mises only**; `StressTensor`, `Facets` and `u` are commented out
in the authors' loader, so a displacement field would need the raw tars.

In [ ]:
ROOT = "/content/sfem/Pytorch_Geometric_Files"
if not os.path.exists(f"{ROOT}/processed/val_data.pt"):
    from huggingface_hub import snapshot_download
    print("downloading ~12 GB ...")
    snapshot_download(repo_id="cmudrc/SFEM", repo_type="dataset",
                      allow_patterns=["Pytorch_Geometric_Files/processed/*"],
                      local_dir="/content/sfem", max_workers=8)
print("data ready")

## 3 · Pickle shim

The released `.pt` files were pickled where the normaliser class lived in a
module named `utils`. Pickle stores that import path, and every graph carries a
`normalizer`, so the module must resolve before `torch.load` can rebuild them.
The authors' published `dataloader.py` defines the class inline, so the
repository alone does not reveal this.

In [ ]:
pathlib.Path("/content/utils.py").write_text("""
import torch
class UnitGaussianNormalizer(object):
    def __init__(self, x=None, eps=1e-5):
        if x is not None:
            self.mean = torch.mean(x, 0); self.std = torch.std(x, 0)
        self.eps = eps
    def encode(self, x): return (x - self.mean) / (self.std + self.eps)
    def decode(self, x, i=None): return x * (self.std + self.eps) + self.mean
class _Ghost:
    def __init__(self, *a, **k): pass
    def __setstate__(self, s):
        if isinstance(s, dict): self.__dict__.update(s)
def __getattr__(n): return type(n, (_Ghost,), {})
""")
if "/content" not in sys.path: sys.path.insert(0, "/content")
sys.modules.pop("utils", None); import utils
print("shim ready")

## 4 · Load and split

`val` is used for the dry run because `train_data.pt` is 10 GB and will not fit
in a standard Colab instance. Switch `SPLIT` to `"train"` on a high-RAM runtime.

In [ ]:
from torch_geometric.data import InMemoryDataset
from torch_geometric.transforms import Compose, Cartesian, PointPairFeatures

SPLIT = "val"          # "train" needs a high-RAM runtime

class SFEM(InMemoryDataset):
    def __init__(self, root, split, transform=None):
        self.split = split
        super().__init__(root, transform)
        self.data, self.slices = torch.load(
            self.processed_paths[0], weights_only=False)
    @property
    def raw_file_names(self): return []
    @property
    def processed_file_names(self): return [f"{self.split}_data.pt"]
    def download(self): pass
    def process(self): pass

full = SFEM(ROOT, SPLIT, transform=Compose([Cartesian(), PointPairFeatures()]))
idx = torch.randperm(len(full), generator=torch.Generator().manual_seed(0))
k = int(0.8 * len(full))
train_ds, val_ds = full[idx[:k]], full[idx[k:]]
print(f"train {len(train_ds)}  calibration {len(val_ds)}"
      f"  x columns {full[0].x.shape[1]}")

## 5 · Dimensionless target

`tau = sigma * L^2 / F`, with `L` the bounding-box diagonal and `F` recovered
from the load one-hot. Dimensionless, so the same number applies in any
consistent unit system — train in SI, predict in mm/N/MPa.

In [ ]:
LOADS_N = torch.tensor([200.0, 2000.0, 20000.0])   # small, medium, large

def char_length(pos):
    return (pos.max(0).values - pos.min(0).values).norm()

def nondim(d):
    """sigma * L^2 / F for one graph."""
    F = LOADS_N[d.x[0, -3:].argmax()]
    return d.y_original.reshape(-1) * char_length(d.pos) ** 2 / F

def nondim_batch(batch):
    """Same, vectorised over a mini-batch."""
    y, bi = batch.y_original.reshape(-1), batch.batch
    loads = LOADS_N.to(y.device)
    L = torch.zeros(batch.num_graphs, device=y.device)
    F = torch.zeros(batch.num_graphs, device=y.device)
    for g in range(batch.num_graphs):
        m = bi == g
        L[g] = char_length(batch.pos[m])
        F[g] = loads[batch.x[m][0, -3:].argmax()]
    return y * (L[bi] ** 2) / F[bi]

s = torch.log1p(torch.cat([nondim(train_ds[i]) for i in range(800)]))
G_MEAN, G_STD = s.mean().item(), s.std().item()
encode = lambda t: (torch.log1p(t) - G_MEAN) / (G_STD + 1e-5)
decode = lambda t: torch.expm1(t * (G_STD + 1e-5) + G_MEAN)

c = nondim(train_ds[0])
print(f"normaliser mean={G_MEAN:.4f} std={G_STD:.4f}")
print(f"tau  min {c.min():.4g}  median {c.median():.4g}  max {c.max():.4g}")

## 6 · Baseline to beat

Geometric features only. It cannot see *where* the load lands, which for peak
stress is most of the answer — so if the GNN does not clear this comfortably,
the GNN is the problem.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

def feats(d):
    e = d.pos.max(0).values - d.pos.min(0).values
    return torch.cat([e, e.prod().reshape(1),
                      torch.tensor([float(d.pos.shape[0])]),
                      d.x[0, -3:]]).numpy()

def xy(ds, m):
    m = min(m, len(ds))
    return (np.stack([feats(ds[i]) for i in range(m)]),
            np.log1p(np.array([nondim(ds[i]).max().item() for i in range(m)])))

Xtr, ytr = xy(train_ds, 2000); Xva, yva = xy(val_ds, 600)
base = GradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
print(f"BASELINE log-space MAE {mean_absolute_error(yva, base.predict(Xva)):.4f}")

## 7 · Save location

In [ ]:
try:
    from google.colab import drive; drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/fea-surrogate'
except Exception:
    OUT = '/content'
os.makedirs(OUT, exist_ok=True)
CKPT = f'{OUT}/stress_gnn.pt'
print("saving to", CKPT)

## 8 · Model

In [ ]:
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

dev = "cuda" if torch.cuda.is_available() else "cpu"

class StressGNN(nn.Module):
    def __init__(self, in_dim, hidden=128, layers=5):
        super().__init__()
        self.inp = nn.Linear(in_dim, hidden)
        self.convs = nn.ModuleList([SAGEConv(hidden, hidden) for _ in range(layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])
        self.out = nn.Sequential(nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, 1))
    def forward(self, d):
        h = self.inp(d.x)
        for c, n in zip(self.convs, self.norms):
            h = h + torch.nn.functional.silu(n(c(h, d.edge_index)))
        return self.out(h)

model = StressGNN(train_ds[0].x.shape[1]).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
tl = DataLoader(train_ds, batch_size=8, shuffle=True)
vl = DataLoader(val_ds, batch_size=8)
print(f"in_dim={train_ds[0].x.shape[1]} device={dev}")

## 9 · Train

Checkpoints every epoch, so a Colab disconnect costs one epoch rather than the
run.

In [ ]:
EPOCHS = 8
for ep in range(EPOCHS):
    model.train(); tot = seen = 0
    for b in tl:
        b = b.to(dev)
        opt.zero_grad()
        loss = torch.nn.functional.smooth_l1_loss(
            model(b), encode(nondim_batch(b)).reshape(-1, 1))
        loss.backward(); opt.step()
        tot += loss.item() * b.num_graphs; seen += b.num_graphs
    torch.save(model.state_dict(), CKPT)
    print(f"epoch {ep+1}/{EPOCHS}  train {tot/seen:.4f}")

## 10 · Calibrate and export

Conformal on log residuals, so the band is a factor rather than a width. The
quintile table is the part worth reading: marginal coverage can be exactly right
while conditional coverage fails badly at one end of the range.

In [ ]:
model.eval(); preds, truths = [], []
with torch.no_grad():
    for b in vl:
        b = b.to(dev)
        p = decode(model(b).reshape(-1)).cpu()
        t = nondim_batch(b).cpu()
        bi = b.batch.cpu()
        for g in range(b.num_graphs):
            m = bi == g
            preds.append(p[m].max().item()); truths.append(t[m].max().item())

preds, truths = np.array(preds), np.array(truths)
ALPHA, n = 0.10, len(preds)
qlvl = min(1.0, np.ceil((n + 1) * (1 - ALPHA)) / n)

# log, not log1p. log1p(x) ~ x for small x, so the band turns additive at the
# bottom of the range and multiplicative at the top: on synthetic data that
# gives quintile coverage [1.00, 1.00, 0.90, 0.76, 0.80] while the marginal
# number still reads 0.89. Plain log holds ~0.90 in every quintile. A peak is a
# maximum over a graph and is never zero, so log is safe here even though the
# per-node normaliser above needs log1p.
lg = lambda v: np.log(np.maximum(v, 1e-12))
res_log = np.abs(lg(preds) - lg(truths))
Q_LOG = float(np.quantile(res_log, qlvl, method="higher"))
lo = np.exp(lg(preds) - Q_LOG)
hi = np.exp(lg(preds) + Q_LOG)
cov = float(((truths >= lo) & (truths <= hi)).mean())
print(f"n={n}  band x/÷ {np.exp(Q_LOG):.2f}  coverage {cov:.3f}")

order = np.argsort(truths)
print("coverage by peak-size quintile:")
for i in range(5):
    sl = order[i * n // 5:(i + 1) * n // 5]
    q = float(((truths[sl] >= lo[sl]) & (truths[sl] <= hi[sl])).mean())
    print(f"  {i+1}: {q:.2f}")

rec = {
    "model": {"name": "sfem-mesh-gnn", "version": "0.2.0",
              "sha256": hashlib.sha256(open(CKPT, 'rb').read()).hexdigest()},
    "coverage": 1 - ALPHA,
    "empirical_coverage": cov,
    "calibration": {"method": "split_conformal_log_residual",
                    "set_size": n,
                    "set_sha256": hashlib.sha256(res_log.tobytes()).hexdigest(),
                    "log_half_width": Q_LOG,
                    "band_factor": float(np.exp(Q_LOG))},
    "target": {"quantity": "von_mises_peak",
               "form": "dimensionless tau = sigma * L^2 / F",
               "L": "bounding-box diagonal",
               "to_physical": "sigma[MPa] = tau * F[N] / L[mm]^2"},
    "normaliser": {"log1p_mean": G_MEAN, "log1p_std": G_STD},
    "domain": {"material_class": "linear_elastic",
               "poisson_ratio": 0.4002,
               "load_pattern": "clumped point loads on facets",
               "source_dataset": "cmudrc/SFEM",
               "split_used": SPLIT},
    "limitations": [
        "Poisson's ratio 0.4002 only; steel at ~0.3 is an extrapolation",
        "loads are clumped point loads, not distributed or contact",
        "synthetic BrepGen geometry; no gear pairs or shafts",
        "screening estimate, not a strength certification",
    ],
    "trained_at": datetime.datetime.now(datetime.UTC).isoformat(),
}
json.dump(rec, open(f"{OUT}/calibration.json", "w"), indent=2)
print(json.dumps(rec, indent=2))

## What comes back

`stress_gnn.pt` and `calibration.json` — megabytes. The dataset stays here.

`fea-surrogate` loads both, meshes an incoming STEP file, builds the same node
features, and emits `SurrogateScreening/v1`. The CAD Agent side already accepts
that document, checks the query against `domain`, and refuses it when a design
falls outside — which, given `poisson_ratio` and the load pattern above, it
frequently should.
